# 04 - BERTopic Topic Discovery

**Objective**: Discover research topics using UMAP + HDBSCAN + c-TF-IDF (BERTopic-style).

**Input**:
- `outputs/mhealth_preprocessed.csv`
- `outputs/document_embeddings.npy`
- `outputs/document_embedding_index.csv`

**Outputs**:
- `outputs/topic_assignments.csv`
- `outputs/topic_summary.csv`
- `outputs/topic_year_trends.csv`
- `outputs/topic_top_terms.csv`
- `figures/topic_frequency.png`

In [1]:
import sys
from pathlib import Path

project_root = Path().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

In [2]:
import numpy as np
import pandas as pd
from src.config_loader import set_seed, ensure_dirs
from src.topic_modeling import run_topic_modeling

dirs = ensure_dirs()
set_seed()

[CONFIG] Random seed set to 42


## Load Data and Embeddings

In [3]:
df = pd.read_csv(dirs['outputs'] / 'mhealth_preprocessed.csv', encoding='utf-8-sig')
embeddings = np.load(dirs['outputs'] / 'document_embeddings.npy')
print(f'Records: {len(df):,}, Embeddings: {embeddings.shape}')

Records: 12,642, Embeddings: (12642, 768)


## Run Topic Modeling

In [4]:
topic_results = run_topic_modeling(df, embeddings, text_column='clean_document')

2026-09-26 17:43:35,334 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


[CONFIG] Random seed set to 42

OFFICIAL BERTopic TOPIC MODELING

  Documents   : 12,642
  Embeddings  : (12642, 768)
  UMAP        : n_components=5  n_neighbors=15  metric=cosine
  HDBSCAN     : min_cluster_size=30  min_samples=10
  n-gram      : (1, 3)
  Random seed : 42

[BERTopic] Fitting model on precomputed embeddings...


2026-09-26 17:44:21,711 - BERTopic - Dimensionality - Completed ✓


2026-09-26 17:44:21,713 - BERTopic - Cluster - Start clustering the reduced embeddings


2026-09-26 17:44:22,287 - BERTopic - Cluster - Completed ✓


2026-09-26 17:44:22,297 - BERTopic - Representation - Fine-tuning topics using representation models.


2026-09-26 17:44:45,271 - BERTopic - Representation - Completed ✓



[BERTopic] === Topic Discovery Results ===
  Topics found         : 63
  Outlier docs (id=-1) : 4,715  (37.3%)
  Docs in topics       : 7,927

  Top 10 topics by document count:
    Topic   0:  683 docs — health, maternal, workers, women, community, mobile, care, services, m
    Topic   1:  634 docs — hiv, men, prep, sexual, intervention, adherence, living hiv, sex, art,
    Topic   2:  479 docs — health, digital, research, data, mhealth, privacy, digital health, app
    Topic   3:  366 docs — apps, quality, app, store, mars, rating, google, stores, available, go
    Topic   4:  287 docs — cancer, patients, breast, breast cancer, survivors, cancer patients, a
    Topic   5:  267 docs — smoking, cessation, smoking cessation, quit, smokers, tobacco, abstine
    Topic   6:  230 docs — pain, knee, chronic, patients, chronic pain, exercise, app, low pain, 
    Topic   7:  226 docs — intention, acceptance, use, perceived, model, intention use, mhealth, 
    Topic   8:  215 docs — children, 


[SAVE] Topic assignments -> E:\Major_Project_Implementation\outputs\topic_assignments.csv  (12,642 rows)
[SAVE] Topic summary     -> E:\Major_Project_Implementation\outputs\topic_summary.csv  (64 topics)
[SAVE] Topic top terms   -> E:\Major_Project_Implementation\outputs\topic_top_terms.csv  (945 rows)
[SAVE] Topic-year matrix -> E:\Major_Project_Implementation\outputs\topic_year_matrix.csv  (11, 63)
[SAVE] BERTopic model    -> E:\Major_Project_Implementation\models\bertopic_model


## Topic Summary

In [5]:
topic_results['summary']

,topic_id,count,percentage,top_terms,representative_titles
0,-1,4715,37.30,(outlier / unassigned),
1,0,683,5.40,"health, maternal, workers, women, community, m...",Mobile phone based clinical guidance for rural...
2,1,634,5.02,"hiv, men, prep, sexual, intervention, adherenc...",Communication technology use and mHealth accep...
3,2,479,3.79,"health, digital, research, data, mhealth, priv...",Strategies for mHealth research: lessons from ...
4,3,366,2.90,"apps, quality, app, store, mars, rating, googl...",A content analysis of smartphone based applica...
...,...,...,...,...,...
59,58,34,0.27,"eye, visual, vision, visual acuity, acuity, sm...",Evaluation of tablet computers for visual func...
60,59,34,0.27,"heart rhythm, rhythm, article, society, heart,...",Correction of: Sleep Quality Prediction From W...
61,60,31,0.25,"diabetes, type diabetes, t2d, activity, type, ...",Acceptability of smartphone technology to inte...
62,61,30,0.24,"mental, mental health, health, workers, interv...",Opportunities and barriers in service delivery...


## Verify Saved BERTopic Model Reloading

Verify that the trained model saved to `models/bertopic_model/` can be reloaded and queried.

In [6]:
from src.topic_modeling import load_saved_bertopic_model

# Reload model from disk
loaded_model = load_saved_bertopic_model()
topic_info = loaded_model.get_topic_info()
print('Loaded model topics:', len(loaded_model.get_topics()) - 1)
print('Topic 0 top terms:', loaded_model.get_topic(0)[:5])
topic_info.head(10)

2026-09-26 17:44:51,051 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


[LOAD] Loading BERTopic model from: E:\Major_Project_Implementation\models\bertopic_model


Loaded model topics: 63
Topic 0 top terms: [['health', 0.017815520067329827], ['maternal', 0.016882333584613784], ['workers', 0.011731672549596756], ['women', 0.01100956740963923], ['community', 0.010954476230955848]]


,Topic,Count,Name,Representation,Representative_Docs
0,-1,4715,-1_health_study_app_mobile,"[health, study, app, mobile, patients, mhealth...",NaN
1,0,683,0_health_maternal_workers_women,"[health, maternal, workers, women, community, ...",NaN
2,1,634,1_hiv_men_prep_sexual,"[hiv, men, prep, sexual, intervention, adheren...",NaN
3,2,479,2_health_digital_research_data,"[health, digital, research, data, mhealth, pri...",NaN
4,3,366,3_apps_quality_app_store,"[apps, quality, app, store, mars, rating, goog...",NaN
5,4,287,4_cancer_patients_breast_breast cancer,"[cancer, patients, breast, breast cancer, surv...",NaN
6,5,267,5_smoking_cessation_smoking cessation_quit,"[smoking, cessation, smoking cessation, quit, ...",NaN
7,6,230,6_pain_knee_chronic_patients,"[pain, knee, chronic, patients, chronic pain, ...",NaN
8,7,226,7_intention_acceptance_use_perceived,"[intention, acceptance, use, perceived, model,...",NaN
9,8,215,8_children_parents_obesity_intervention,"[children, parents, obesity, intervention, ado...",NaN
